# Measuring the Influence of Trump's Tweets on Market Volatility and Direction

**Notebook Outline**

- Notebook Setup
- Data Collection
- Sentiment Analysis
- Feature Engineering
- Regression Model

## Notebook Setup

In [48]:
import os
import requests
import zipfile
from dotenv import load_dotenv
import json
import pandas as pd
import numpy as np
import yfinance as yf
import vaderSentiment as vd

!pip install -q kaggle



[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Data Collection

### Financial Market Data from Yahoo Finance
Using yfinance, Yahoo Finance's API wrapper, we are able to pull historical records of two major indexes. The first, VIX, is an index that attempts to capture volatility by measuring the magnitude of price moevments in the market. The second, SPY, is technically a fund that tracks all the companies in the S&P 500, but is often used as a benchmark to measure "the (U.S.) market" in finance. 


In [49]:
# download() from yfinance pulls in historical records for the specified ticker symbol and date range.

vix = yf.download('^VIX', start='2025-01-01', end='2026-3-31')
spy = yf.download('SPY', start='2025-01-01', end='2026-03-31')


[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  1 of 1 completed


In [50]:
vix.head(10)

Price,Close,High,Low,Open,Volume
Ticker,^VIX,^VIX,^VIX,^VIX,^VIX
Date,,,,,
2025-01-02,17.930000,19.500000,16.959999,17.209999,0
2025-01-03,16.129999,17.940001,16.110001,17.660000,0
2025-01-06,16.040001,16.870001,15.710000,16.770000,0
2025-01-07,17.820000,18.900000,15.790000,16.480000,0
2025-01-08,17.700001,19.500000,17.370001,17.910000,0
2025-01-10,19.540001,20.309999,18.049999,18.290001,0
2025-01-13,19.190001,22.040001,19.150000,21.180000,0
2025-01-14,18.709999,19.660000,18.240000,18.790001,0


In [51]:
spy.head(10)

Price,Close,High,Low,Open,Volume
Ticker,SPY,SPY,SPY,SPY,SPY
Date,,,,,
2025-01-02,576.280273,582.677464,572.199457,580.962353,50204000
2025-01-03,583.485779,584.126448,578.044689,579.128997,37888500
2025-01-06,586.847107,591.125077,585.112263,587.744129,47679400
2025-01-07,580.213318,589.202908,578.389795,588.877610,60393100
2025-01-08,581.060974,582.135415,576.832338,580.282292,47304700
2025-01-10,572.189636,577.571586,570.277374,577.502580,73105000
2025-01-13,573.076904,573.431742,567.123230,567.537268,47910100
2025-01-14,573.865356,576.635174,570.080238,576.004311,48420600


### Pull Data from Truth Social
This section authenticates access with the GitHub API using a Personal Access Token. With authenticated access, we retrive metadata from the truth_archive.json file from the repository at stiles/trump-truth-social-archive. Then with the direct download URL in the metadata, we download the file into our project. After filtering out duplicates and null values in key rows during the cleaning process, we have a dataframe of 8 columns and ~5,000 rows.

In [52]:
# AUTHENTICATE WITH GITHUB API TO PULL DOWNLOAD URL

load_dotenv()  # Load hiden access token from .env file
ACCESS_TOKEN = os.getenv("GITHUB_TOKEN")

headers = {"Authorization": f"Bearer {ACCESS_TOKEN}"}
url = "https://api.github.com/repos/stiles/trump-truth-social-archive/contents/data/truth_archive.json"

response = requests.get(url, headers=headers) # Make the request to GitHub
print("STATUS:", response.status_code)
data = response.json()


# USING THE AUTHENTICATED URL, DOWNLOAD THE TRUTH SOCIAL DATASET

download_url = data["download_url"] # Pull the download url from the response
file_response = requests.get(download_url)

with open("truth_archive.json", "wb") as f:
    f.write(file_response.content)

print("Truth Social dataset downloaded successfully!")


# LOAD THE TRUTH SOCIAL DATASET INTO A PANDAS DATAFRAME

with open("truth_archive.json", "r", encoding="utf-8") as f:
    truth_data = json.load(f)

truth_df = pd.DataFrame(truth_data)
print(truth_df.shape)
truth_df.head()


STATUS: 200
Truth Social dataset downloaded successfully!
(29469, 8)


,id,created_at,content,url,media,replies_count,reblogs_count,favourites_count
0,116507513607934090,2026-05-02T23:12:54.339Z,"I hate seeing Fox, and other Conservative Outl...",https://truthsocial.com/@realDonaldTrump/11650...,[],3752.0,5482.0,22203.0
1,116507501775457290,2026-05-02T23:09:53.790Z,So ironic that Cryinâ Chuck Schumer and the ...,https://truthsocial.com/@realDonaldTrump/11650...,[],2524.0,7084.0,23750.0
2,116507414650995614,2026-05-02T22:47:44.377Z,I will soon be reviewing the plan that Iran ha...,https://truthsocial.com/@realDonaldTrump/11650...,[],3015.0,5610.0,23285.0
3,116502923327437911,2026-05-02T03:45:32.282Z,,https://truthsocial.com/@realDonaldTrump/11650...,[https://static-assets-1.truthsocial.com/tmtg:...,3204.0,6145.0,31912.0
4,116502906556000308,2026-05-02T03:41:16.371Z,,https://truthsocial.com/@realDonaldTrump/11650...,[https://static-assets-1.truthsocial.com/tmtg:...,865.0,3638.0,22701.0


In [53]:
# CLEAN TRUTH SOCIAL DATA

truth_df = truth_df.dropna(subset=["content"]).copy()

truth_df["created_at"] = pd.to_datetime(
    truth_df["created_at"],
    errors="coerce"
)

truth_df = truth_df.dropna(subset=["content"])      # Key requirement for analysis
truth_df = truth_df.dropna(subset=["created_at"])   # Key requirement for analysis
print(truth_df.shape)

Posts_df_truth = truth_df[truth_df['created_at']>= '2025-01-01'] # The month of Trump's first term
print(Posts_df_truth.shape)
Posts_df_truth.head(10)

(29469, 8)
(5088, 8)


,id,created_at,content,url,media,replies_count,reblogs_count,favourites_count
0,116507513607934090,2026-05-02 23:12:54.339000+00:00,"I hate seeing Fox, and other Conservative Outl...",https://truthsocial.com/@realDonaldTrump/11650...,[],3752.0,5482.0,22203.0
1,116507501775457290,2026-05-02 23:09:53.790000+00:00,So ironic that Cryinâ Chuck Schumer and the ...,https://truthsocial.com/@realDonaldTrump/11650...,[],2524.0,7084.0,23750.0
2,116507414650995614,2026-05-02 22:47:44.377000+00:00,I will soon be reviewing the plan that Iran ha...,https://truthsocial.com/@realDonaldTrump/11650...,[],3015.0,5610.0,23285.0
3,116502923327437911,2026-05-02 03:45:32.282000+00:00,,https://truthsocial.com/@realDonaldTrump/11650...,[https://static-assets-1.truthsocial.com/tmtg:...,3204.0,6145.0,31912.0
4,116502906556000308,2026-05-02 03:41:16.371000+00:00,,https://truthsocial.com/@realDonaldTrump/11650...,[https://static-assets-1.truthsocial.com/tmtg:...,865.0,3638.0,22701.0
5,116502902374252861,2026-05-02 03:40:12.563000+00:00,"This is what our Country was before, and after...",https://truthsocial.com/@realDonaldTrump/11650...,[https://static-assets-1.truthsocial.com/tmtg:...,2272.0,6451.0,29775.0
6,116502893543384666,2026-05-02 03:37:57.815000+00:00,,https://truthsocial.com/@realDonaldTrump/11650...,[https://static-assets-1.truthsocial.com/tmtg:...,1390.0,3424.0,20605.0
7,116502871108388206,2026-05-02 03:32:15.483000+00:00,,https://truthsocial.com/@realDonaldTrump/11650...,[https://static-assets-1.truthsocial.com/tmtg:...,925.0,4152.0,26079.0
8,116502848851907459,2026-05-02 03:26:35.877000+00:00,,https://truthsocial.com/@realDonaldTrump/11650...,[https://static-assets-1.truthsocial.com/tmtg:...,2445.0,5879.0,30410.0
9,116502833969390552,2026-05-02 03:22:48.789000+00:00,,https://truthsocial.com/@realDonaldTrump/11650...,[https://static-assets-1.truthsocial.com/tmtg:...,1096.0,3913.0,22643.0


### Pull Data from Kaggle
In this section we make an authenticated call to the Kaggle API. With access we pull truth social data from Kaggle and compare with the GitHub file. Note that while the dataste is named 'trump_tweets_data' we filter for only Truth Social data during the cleaning process. The result is a dataset of 18 columns and ~6,500 rows. We chose to move forward with the data from Kaggle.

In [ ]:
# AUTHENTICATE WITH KAGGLE API

load_dotenv()  # Load hiden access token from .env file
KAGGLE_USERNAME = os.getenv("KAGGLE_USERNAME")
KAGGLE_KEY = os.getenv("KAGGLE_KEY")

kaggle_dir = os.path.expanduser("~/.kaggle") # Creates a hidden directory required for storing API credentials
os.makedirs(kaggle_dir, exist_ok=True)

kaggle_path = os.path.join(kaggle_dir, "kaggle.json")

if os.path.exists(kaggle_path):   
    print("kaggle.json already exists — skipping creation.") # Avoid overwriting existing credentials
    
else:
    kaggle_credentials = {
        "username": KAGGLE_USERNAME,
        "key": KAGGLE_KEY
    }
    with open(kaggle_path, "w") as f:
        json.dump(kaggle_credentials, f)

    print("kaggle.json created successfully!")


# USING THE AUTHENTICATED KAGGLE API, DOWNLOAD THE TRUMP TWEETS DATASET

!kaggle datasets download -d datadrivendecision/trump-tweets-2009-2025

zip_path = "trump-tweets-2009-2025.zip"
extract_path = "trump_tweets_data"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")

#Check which files are inside
files = os.listdir(extract_path)
print(files)

# Load the CSV file
csv_path = os.path.join(extract_path, files[0])
df = pd.read_csv(csv_path)


# CLEAN KAGGLE TRUMP TWEETS DATA

df = df.dropna(subset=["text"]).copy()
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["date"])
Kaggle_df = df[df['platform'] != "Twitter"]
Posts_df_kaggle = Kaggle_df[Kaggle_df['date'] >='2025-01-01']

print(Posts_df_kaggle.shape)
Posts_df_kaggle


kaggle.json already exists — skipping creation.
Dataset URL: https://www.kaggle.com/datasets/datadrivendecision/trump-tweets-2009-2025
License(s): CC0-1.0
trump-tweets-2009-2025.zip: Skipping, found more recently modified local copy (use --force to force download)
Dataset extracted successfully!
['djt_posts_dec2025.csv']
(6497, 18)


,id,date,platform,handle,text,favorite_count,repost_count,quote_flag,repost_flag,deleted_flag,word_count,hashtags,urls,user_mentions,media_count,media_urls,post_url,in_reply_to
0,115816402893182666,2025-12-31 21:54:21+00:00,Truth Social,realDonaldTrump,"Good News! George and Amal Clooney, two of the...",52,16,False,False,False,144,NaN,NaN,NaN,0,NaN,https://truthsocial.com/@realDonaldTrump/posts...,NaN
1,115816171466225987,2025-12-31 20:55:30+00:00,Truth Social,realDonaldTrump,We are removing the National Guard from Chicag...,44,16,False,False,False,107,NaN,NaN,NaN,0,NaN,https://truthsocial.com/@realDonaldTrump/posts...,NaN
2,115816098525451632,2025-12-31 20:36:57+00:00,Truth Social,realDonaldTrump,The Democrats are a bunch of cheaters and thie...,461,179,False,False,False,50,NaN,NaN,NaN,0,NaN,https://truthsocial.com/@realDonaldTrump/posts...,NaN
3,115816063544553030,2025-12-31 20:28:03+00:00,Truth Social,realDonaldTrump,Republicans: No more money to Fat Cat Insuranc...,309,97,False,False,False,24,NaN,NaN,NaN,0,NaN,https://truthsocial.com/@realDonaldTrump/posts...,NaN
4,115815971019162695,2025-12-31 20:04:31+00:00,Truth Social,realDonaldTrump,The United States has set a World Record on in...,36,11,False,False,False,76,NaN,NaN,NaN,0,NaN,https://truthsocial.com/@realDonaldTrump/posts...,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6639,113755111968756066,2025-01-01 21:00:58+00:00,Truth Social,realDonaldTrump,https://www. bostonglobe.com/2024/12/31/opi ni...,2681,759,False,False,False,3,NaN,NaN,NaN,0,NaN,https://truthsocial.com/@realDonaldTrump/posts...,NaN
6640,113754860982199539,2025-01-01 19:57:08+00:00,Truth Social,realDonaldTrump,[Video],11239,2509,False,False,False,1,NaN,NaN,NaN,1,NaN,https://truthsocial.com/@realDonaldTrump/posts...,NaN
6641,113754577369839812,2025-01-01 18:45:00+00:00,Truth Social,realDonaldTrump,[Image],15038,3205,False,False,False,1,NaN,NaN,NaN,1,NaN,https://truthsocial.com/@realDonaldTrump/posts...,NaN
6642,113754467479053315,2025-01-01 18:17:04+00:00,Truth Social,realDonaldTrump,Happy New Year to all. It will be a great time...,18069,3646,False,False,False,13,NaN,NaN,NaN,0,NaN,https://truthsocial.com/@realDonaldTrump/posts...,NaN


## Sentiment Analysis
In our case, the raw text data isn't very valuable. In this section, we transform the text values into quantitative measures of sentiment using tools from the vaderSentiment library. (More notes on vaderSentiment...). (Notes on the process of assigning sentiment values to Tweets...).

In [ ]:
### To score tweets based on sentiment looks quite doable with SentimentIntensityAnlayzer(). Example usage is illustrated in the documentation;
### https://vadersentiment.readthedocs.io/en/latest/pages/code_and_example.html

## Feature Engineering
In this section, we transform our collected data so that is is more meaningful to our model, we align the various collections to fit together in a dataset, and build experiemental features. 

In [ ]:
### Take the returns of the SPY and VIX series rather than their closing values.

df = pd.DataFrame()
df['SPY_Returns'] = spy['Close'].pct_change()
df['VIX_Returns'] = vix['Close'].pct_change()

print(df.head())

### Aggregate the number of tweets per day, and the average sentiment score per day.
### Create experiemental features such as lagged returns, lagged sentiment scores, and rolling averages of sentiment scores.

            SPY_Returns  VIX_Returns
Date                                
2025-01-02          NaN          NaN
2025-01-03     0.012503    -0.100390
2025-01-06     0.005761    -0.005580
2025-01-07    -0.011304     0.110972
2025-01-08     0.001461    -0.006734


## Estimation Model
This section implements our framework for quantifying the relationship between tweet-derived features and stock market outcomes.

In [ ]:
### We can first plot some of the data to get a sense of whether or not we should use simpl linear regression or a more complex non-linear model.
### We can also use a base model/advanced model approach and evaluate based on the differences bwetween the two.
### If estimative power seems to be rather poor, we can add macroeconomic indicators into the model.